# SpectraRestore — Google Colab Training & Evaluation Pipeline

**SEMICON India Hackathon 2026 · KLA Track 1 (PS01)**  
**Team:** ChipSync · SRM Institute of Science And Technology  
**Architecture:** Joint Blind Denoising + 2× Super-Resolution for SEM Images (`NAFNet-SR2×`)

---

### Pipeline Design & Safety Guarantees
1. **Zero-Tolerance Gates:** Execution will immediately halt if GPU is unavailable, dataset pairing fails pre-flight checks, LPIPS fails initialization, or smoke tests fail.
2. **Quick & Full Recipes:** Configurable between **Quick Test** (`2,000` iters for immediate end-to-end verification) and **Full Training** (`200,000` iters for full convergence).
3. **Strict Validation vs Blind Test Separation:** Validation outputs (`outputs/val_restored`) and competition test outputs (`outputs/test_restored`) are strictly isolated.
4. **Complete Artifact Generation:** Generates multi-zoom visual evidence (`outputs/visual_evidence.png`), full reproducibility log (`outputs/reproducibility.txt`), Slide 6 results table, and verified submission package (`submission_SpectraRestore.zip`).

## 1 · Environment & GPU Hard Stop

Verifies CUDA hardware acceleration, PyTorch version, GPU name, and VRAM. **Halts execution if GPU is not active.**

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "\n❌ GPU ACCELERATION REQUIRED — Training blocked!\n"
    "Please enable a GPU runtime: Runtime > Change runtime type > Hardware accelerator > GPU (T4 / A100 / L4)."
)

gpu_props = torch.cuda.get_device_properties(0)
vram_gb = gpu_props.total_memory / (1024 ** 3)

print("=" * 60)
print("[environment pre-flight]")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()} (CUDA {torch.version.cuda})")
print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
print(f"Total VRAM:      {vram_gb:.2f} GB")
print("=" * 60)

## 2 · Clone Repository from GitHub

Clones the public repository directly as the sole source of truth.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Charan-suresh/SpectraRestore.git"
PROJECT = Path("/content/SpectraRestore")

if (PROJECT / "src" / "model.py").is_file():
    print(f"Project repository found at {PROJECT} — pulling latest updates...")
    subprocess.run(["git", "pull", "origin", "main"], cwd=PROJECT, check=True)
else:
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

%cd {PROJECT}
assert (PROJECT / "src" / "model.py").is_file(), "Setup failed — repository cloning was unsuccessful."
print(f"\nActive Workspace: {Path.cwd()}")

## 3 · Install Dependencies & Verify LPIPS (Hard Gate)

Installs project requirements and strictly tests that `lpips` pretrained weights initialize properly.

In [ ]:
%pip install -q -r requirements.txt

import sys
from pathlib import Path
PROJECT = Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Hard verification of LPIPS pretrained perceptual metric
import lpips
try:
    lp_checker = lpips.LPIPS(net="alex")
    dummy_a = torch.zeros(1, 3, 64, 64)
    dummy_b = torch.zeros(1, 3, 64, 64)
    _ = lp_checker(dummy_a, dummy_b)
    print("=" * 60)
    print("[check] LPIPS initialization: PASS (AlexNet perceptual model active)")
    print("=" * 60)
except Exception as e:
    raise RuntimeError(
        f"❌ LPIPS initialization failed: {e}\n"
        "Training has been blocked because perceptual loss and evaluation metric cannot be verified."
    )

## 4 · Model Architecture & Parameter Count Verification

Validates model architecture presets against actual code implementations.

In [ ]:
from src.model import build_model

m_default = build_model("default")
m_fast = build_model("fast")
m_large = build_model("large")
m_tiny = build_model("tiny")

print("=" * 60)
print("SPECTRARESTORE ARCHITECTURE PRESETS")
print("=" * 60)
print(f"default (width 32, full depth):  {m_default.num_params()/1e6:6.2f}M params  [Competition Target]")
print(f"fast    (width 32, light):       {m_fast.num_params()/1e6:6.2f}M params  [Timing Benchmark Fallback]")
print(f"large   (width 48, full depth):  {m_large.num_params()/1e6:6.2f}M params  [Quality Upper Bound]")
print(f"tiny    (width 16, smoke test):  {m_tiny.num_params()/1e6:6.2f}M params  [Unit Testing]")
print("=" * 60)

## 5 · Dataset Setup & Pre-Flight Check (Hard Gate)

Verifies dataset presence from the repository, auto-extracts archives if provided, creates demo data if missing, and runs strict preflight validation.

In [ ]:
import zipfile
from pathlib import Path
from scripts.dataset_preflight import validate_dataset
from scripts.create_demo_data import create_demo_dataset

data_root = Path("data")
data_root.mkdir(exist_ok=True)

# 1. Auto-extract any uploaded zip file in /content/ if present
if Path("/content").exists():
    for zip_cand in Path("/content").glob("*.zip"):
        if not (data_root / "train" / "degraded").is_dir():
            print(f"[dataset setup] Extracting archive: {zip_cand} -> data/")
            with zipfile.ZipFile(zip_cand, 'r') as zf:
                zf.extractall(data_root)

# 2. Check if dataset exists in repo; if not present, populate it with valid SEM pairs
train_deg = data_root / "train" / "degraded"
train_gt = data_root / "train" / "gt"
if not train_deg.is_dir() or len(list(train_deg.glob("*"))) == 0:
    print("[dataset setup] Dataset not found in repo — generating initial SEM dataset in data/...")
    create_demo_dataset(data_root=data_root, num_train=20, num_val=6)

# 3. Run strict dataset pre-flight validation
counts = validate_dataset(data_root=data_root, scale=2, check_geometry=True)
print(f"[dataset setup] Dataset verified and ready: {counts['train']} train pairs, {counts['val']} val pairs.")

## 6 · Pipeline Smoke Test Gate (Hard Subprocess Check)

Runs an end-to-end integration test validating forward pass, loss computation, gradient backpropagation, 16-bit raster scaling, and inference round-trip.

In [ ]:
import subprocess
import sys

print("[smoke-test gate] Executing scripts/smoke_test.py ...")
res = subprocess.run([sys.executable, "scripts/smoke_test.py"], capture_output=True, text=True)
print(res.stdout)

if res.returncode != 0:
    print(res.stderr, file=sys.stderr)
    raise RuntimeError("❌ SMOKE TEST FAILED — Training has intentionally been blocked.")

print("=" * 60)
print("[smoke-test gate] PASS — Pipeline verified for training.")
print("=" * 60)

## 7 · Training Configuration & Hardware-Aware Batch Sizing

- **Quick Test Recipe (Active):** `QUICK_TEST_ITERS = 2_000` (Fast pipeline test)
- **Full Convergence Recipe:** `FULL_TRAIN_ITERS = 200_000` (Production run)
- **Auto Batch Size:** Scaled to available GPU VRAM (T4: 4, L4: 8, A100: 16)

In [ ]:
import torch
from pathlib import Path

FULL_TRAIN_ITERS = 200_000
QUICK_TEST_ITERS = 2_000

# Active configuration: set to QUICK_TEST_ITERS for immediate verification
# (Switch to FULL_TRAIN_ITERS for final model convergence)
ITERS = QUICK_TEST_ITERS

PRESET = "default"       # default (~29M) | fast (~15M) | large (~65M)
GT_CROP = 256            # 256x256 GT crop (128x128 degraded input)

# GPU-aware batch size selection
def select_adaptive_batch_size(manual_override=None):
    if manual_override is not None:
        return manual_override
    if not torch.cuda.is_available():
        return 2
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if vram >= 38:        # A100 (40GB / 80GB)
        bs = 16
    elif vram >= 20:      # L4 / A10 / RTX 3090 (24GB)
        bs = 8
    elif vram >= 14:      # T4 / V100 (16GB)
        bs = 4
    else:
        bs = 2
    return bs

BATCH_SIZE = select_adaptive_batch_size()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print("=" * 60)
print("[training configuration]")
print(f"Target GPU:       {gpu_name}")
print(f"Model Preset:     {PRESET}")
print(f"Batch Size:       {BATCH_SIZE}")
print(f"Total Iterations: {ITERS:,} {'(Quick Test Recipe)' if ITERS == QUICK_TEST_ITERS else '(Full Recipe)'}")
print(f"GT Crop Size:     {GT_CROP}x{GT_CROP}")
print("=" * 60)

## 8 · Train SpectraRestore (NAFNet-SR2×)

- **Loss Function:** Charbonnier (1.00) + SSIM (0.20) + FFT-L1 (0.05) + LPIPS (0.10 after 20% warmup)
- **Optimization:** AdamW + Cosine Annealing (3e-4 → 1e-6) + BF16 mixed precision
- Checkpoints are saved under `weights/` (`best.pt`, `last_ema.pt`, `ckpt_*.pt`).

In [ ]:
import subprocess
import sys
from pathlib import Path

weights_dir = Path("weights")
weights_dir.mkdir(exist_ok=True)

# Adaptive logging and validation cadence based on total iterations
if ITERS <= 2_000:
    val_every = 200
    save_every = 500
    log_every = 50
else:
    val_every = 2000
    save_every = 5000
    log_every = 100

# Resume from latest checkpoint if available
resume_args = []
ckpts = sorted(weights_dir.glob("ckpt_*.pt"))
if ckpts:
    resume_args = ["--resume", str(ckpts[-1])]
    print(f"[train] Auto-resuming from full checkpoint: {ckpts[-1]}")
elif (weights_dir / "best.pt").is_file():
    resume_args = ["--resume", str(weights_dir / "best.pt")]
    print("[train] Auto-resuming from weights/best.pt")

train_cmd = [
    sys.executable, "-m", "src.train",
    "--data_root", "data",
    "--preset", PRESET,
    "--batch_size", str(BATCH_SIZE),
    "--iters", str(ITERS),
    "--gt_crop", str(GT_CROP),
    "--num_workers", "2",
    "--val_every", str(val_every),
    "--save_every", str(save_every),
    "--log_every", str(log_every),
    "--out_dir", "weights",
] + resume_args

print("Executing training command:")
print(" ".join(train_cmd))
print("-" * 60)

res = subprocess.run(train_cmd)
if res.returncode != 0:
    raise RuntimeError("❌ Training failed or was aborted.")

## 9 · Checkpoint Resolution & Verification

Resolves the best available checkpoint (`best.pt` → `last_ema.pt` → `model.pt`).

In [ ]:
from evaluate import resolve_weights

try:
    selected_weights = resolve_weights(None)
    print("=" * 60)
    print(f"[checkpoint resolved] {selected_weights.resolve()}")
    print(f"File Size: {selected_weights.stat().st_size / (1024**2):.2f} MB")
    print("=" * 60)
except FileNotFoundError as e:
    raise RuntimeError(f"❌ Checkpoint resolution failed: {e}")

## 10 · Validation Evaluation (Standalone KLA evaluate.py)

Evaluates the validation set degraded images and outputs restored images to `outputs/val_restored`.

In [ ]:
import subprocess
import sys
from pathlib import Path

VAL_INPUT = Path("data/val/degraded")
VAL_OUTPUT = Path("outputs/val_restored")

eval_cmd = [
    sys.executable, "evaluate.py",
    "--input_dir", str(VAL_INPUT),
    "--output_dir", str(VAL_OUTPUT),
    "--weights", str(selected_weights),
]

print("Running validation evaluation:")
print(" ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

## 11 · Validation Benchmark & Slide 6 Metrics Table

Computes exact SSIM, pSNR, LPIPS, and inference latency comparing the **Degraded Input Baseline** against **SpectraRestore Output** on the held-out validation split.

In [ ]:
import subprocess
import sys

bench_cmd = [
    sys.executable, "scripts/benchmark_val.py",
    "--data_root", "data",
    "--weights", str(selected_weights),
]

subprocess.run(bench_cmd, check=True)

## 12 · Curated Multi-Zoom Visual Evidence & Error Heatmaps

Generates deterministic 4-panel visual comparisons with **Full-Frame**, **Zoom 1 (2× Crop)**, and **Zoom 2 (4× High-Detail Crop)** for periodic structures, saving to `outputs/visual_evidence.png`.

In [ ]:
from scripts.visual_evidence import generate_visual_evidence
from IPython.display import Image, display

fig_path = generate_visual_evidence(
    restored_dir="outputs/val_restored",
    degraded_dir="data/val/degraded",
    gt_dir="data/val/gt",
    output_path="outputs/visual_evidence.png",
    num_cases=2,
    seed=42,
)

display(Image(filename=str(fig_path)))

## 13 · Competition Blind Test Inference (Strictly Separated from Validation)

Dedicated configuration for the official competition blind test set.  
Outputs are written to `outputs/test_restored` (never overwriting validation results).

In [ ]:
import subprocess
import sys
from pathlib import Path

TEST_INPUT = Path("data/test/degraded")
TEST_OUTPUT = Path("outputs/test_restored")

if TEST_INPUT.is_dir() and len(list(TEST_INPUT.glob("*"))) > 0:
    print(f"[test inference] Running inference on competition test set: {TEST_INPUT} -> {TEST_OUTPUT}")
    test_cmd = [
        sys.executable, "evaluate.py",
        "--input_dir", str(TEST_INPUT),
        "--output_dir", str(TEST_OUTPUT),
        "--weights", str(selected_weights),
    ]
    subprocess.run(test_cmd, check=True)
    print(f"[test inference] Blind test outputs generated in {TEST_OUTPUT}")
else:
    print("[test inference] Official test set not present under 'data/test/degraded'.")
    print("When released by KLA, place test images in data/test/degraded/ and re-run this cell.")

## 14 · Reproducibility Artifact Generation

Captures Git commit hash, working tree status, environment versions, GPU metadata, and complete `pip freeze` package state into `outputs/reproducibility.txt`.

In [ ]:
import datetime
import platform
import subprocess
import sys
from pathlib import Path
import torch

out_file = Path("outputs/reproducibility.txt")
out_file.parent.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True).strip()
    except Exception as e:
        return f"N/A ({e})"
 
git_commit = run_cmd(["git", "rev-parse", "HEAD"])
git_status = run_cmd(["git", "status", "--short"])
pip_freeze = run_cmd([sys.executable, "-m", "pip", "freeze"])

with open(out_file, "w") as f:
    f.write("=" * 60 + "\n")
    f.write("SPECTRARESTORE REPRODUCIBILITY MANIFEST\n")
    f.write("=" * 60 + "\n")
    f.write(f"Timestamp:        {datetime.datetime.now(datetime.timezone.utc).isoformat()}\n")
    f.write(f"Git Commit:       {git_commit}\n")
    f.write(f"Git Status:       {'Clean' if not git_status else 'Dirty: ' + git_status}\n")
    f.write(f"Python Version:   {platform.python_version()} ({sys.executable})\n")
    f.write(f"PyTorch Version:  {torch.__version__}\n")
    f.write(f"CUDA Version:     {torch.version.cuda}\n")
    f.write(f"GPU Device:       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}\n")
    f.write("-" * 60 + "\n")
    f.write("PIP FREEZE ENVIRONMENT:\n")
    f.write("-" * 60 + "\n")
    f.write(pip_freeze + "\n")

print(f"[reproducibility] Manifest saved to {out_file.resolve()}")

## 15 · Submission Checklist & Packaging

Validates the presence of all competition submission deliverables and compiles `submission_SpectraRestore.zip`.

In [ ]:
from scripts.package_submission import create_submission_zip
from pathlib import Path

zip_path = create_submission_zip(Path("submission_SpectraRestore.zip"), strict=False)
print(f"\nSubmission archive ready: {zip_path.resolve()}")